In [ ]:
import re
import pandas as pd

INPUT_FILE = "arsenal_youtube_cafe_after_labeled.xlsx"
OUTPUT_XLSX = "arsenal_youtube_cafe_final_labeled.xlsx"
OUTPUT_CSV = "arsenal_youtube_cafe_final_labeled.csv"

df = pd.read_excel(INPUT_FILE)


def extract_round_from_text(text):
    text = str(text)
    m = re.search(r"(\d{1,2})R", text)
    if m:
        return f"{m.group(1)}R"
    return None

source = df["source"].astype(str).str.lower()

youtube_mask = source.eq("youtube")
cafe_mask = source.eq("cafe")

# title 컬럼 기준으로 YouTube 라운드 재매핑
df.loc[youtube_mask, "round"] = df.loc[youtube_mask, "title"].apply(extract_round_from_text)

# UCL / 챔스 영상은 라운드 분석에서 제외하거나 별도 처리용으로 표시
ucl_mask = (
    youtube_mask
    & df["round"].isna()
    & df["title"].astype(str).str.contains("UCL|챔스|PSG|Champions", case=False, na=False)
)

df.loc[ucl_mask, "round"] = "UCL_FINAL"

# Cafe는 라운드별 데이터가 아니라 38R 전후 비교 데이터
df.loc[cafe_mask, "round"] = "38R_before_after"

# period는 cafe before/after 분석에 쓰이므로 유지
# 혹시 비어 있으면 날짜 기준으로 따로 보완 가능하지만, 지금 파일에는 period가 있는 상태라고 보면 됨

FINAL_COLUMNS = [
    "source",
    "round",
    "club",
    "title",
    "content_type",
    "text_original",
    "text_clean",
    "content_published_at",
    "comment_published_at",
    "like_count",
    "url",
    "period",
    "gpt_sentiment",
    "gpt_confidence",
    "gpt_target",
    "gpt_reason",
    "topic"
]

df_final = df[FINAL_COLUMNS].copy()

print("최종 shape:", df_final.shape)
print("\nsource 분포")
print(df_final["source"].value_counts(dropna=False))

print("\nround 분포")
print(df_final["round"].value_counts(dropna=False).sort_index())

print("\nperiod 분포")
print(df_final["period"].value_counts(dropna=False))

print("\n감정분포")
print(df_final["gpt_sentiment"].value_counts(dropna=False))

df_final.to_excel(OUTPUT_XLSX, index=False)
df_final.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("\n✅ 최종 파일 저장 완료")
print(OUTPUT_XLSX)
print(OUTPUT_CSV)